<a href="https://colab.research.google.com/github/PDM-15/Python-Practice/blob/main/NLP_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Preprocessing

In [50]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

!pip install autocorrect
!pip install pyspellchecker
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 33.5 MB/s eta 0:00:00


In [57]:
import numpy as np
import pandas as pd
import requests  #for fetching data from an api
import string   #required for punctuation removal


from nltk.corpus import stopwords   #required for stop word removal
from nltk import word_tokenize, sent_tokenize
#for spelling correction
#from textblob import TextBlob
#from autocorrect import Speller
from spellchecker import SpellChecker
from nltk.stem import PorterStemmer, WordNetLemmatizer

#TEXT PREPROCESSING -> STEP 1: Data Collection ->TMdb API - movie reviews - json format
url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=471"  #api from where we fetch data
response = requests.get(url) #for getting the data -> gives a status code

if response.status_code == 200:
  df = pd.json_normalize(response.json(), record_path = ['results'])        #convert directly into a dataframe if 200 -> request success, normalize since data is in the form of dictionaries
else:
  raise Exception("API request failed,", response.status_code)              #Throw exception if request denied

print("No. of rows and cols in data: ",df.shape)              #returns the number of rwos and column in the dataframe

df.head(5)            #returns the first five elements of the dataframe

original_data = df['overview'].str.lower()
print("Original data row 1\n",df['overview'][0],"\n")

#sentence tokenizing -> doesn't work after punctuation removal
sentTokenized = df['overview'].apply(nltk.sent_tokenize)
print("After sentence tokenization:\n",sentTokenized[0],"\n")

#STEP 2: Text Cleaning -> HTMLtags, URLs, punctuations, chat word treatment, handling emojis, spelling mistakes

#Removing punctuations and digits
df['overview'] = df['overview'].apply(lambda x: x.translate(str.maketrans('','', string.punctuation + '0123456789')))
print("After removing digits and punctuation:\n",df['overview'][0],"\n")

#Spelling correction  -- commented out because overview is turning null
#df['overview'] = df['overview'].apply(lambda x: str(TextBlob(x).correct())) #--> slower takes several minutes
#df['overview'] = df['overview'].apply(lambda x: Speller(lang = 'en'))
#spell = SpellChecker()
#df['overview'] = df['overview'].apply(lambda x: [spell.correction(word) if word in (unknown_word := spell.unknown(x)) else word for word in x])

#STEP 3: Text Normalization - Lowercasing
df['overview'] = df['overview'].str.lower()
print("After Lowercasing:\n", df['overview'][0],"\n")

#STEP 4: Tokenization

#Word Tokenizing
df['overview'] = df['overview'].apply(nltk.word_tokenize)
print("After word tokenization:\n",df['overview'][0],"\n")

#STEP 5: Stop word removal
stop_word = set(stopwords.words('english'))
df['overview'] = df['overview'].apply(lambda x: [word for word in x if word not in stop_word] )
print("After stop word removal:\n",df['overview'][0],"\n")

#STEP 6: Stemming and Lemmatization

#Stemming -> faster, no semantic present - algorithm based
ps = PorterStemmer()
stem = df['overview'].apply(lambda x: [ps.stem(word) for word in x ])
print("After Stemming: \n", stem[0],"\n")

#Lemmatization -> slower , returns semantic root word - wordnet based
lm = WordNetLemmatizer()
lemma = df['overview'].apply(lambda x: [lm.lemmatize(word) for word in x ])
print("After lemmatization:\n",lemma[0],"\n")


No. of rows and cols in data:  (20, 15)
Original data row 1
 Maya is living the ultimate fashionistas dream: she is working as a stylist for one of the French trend setters, in the capital of haute-couture: Paris.  One of the IT girls of fashion, shes following her dreams until one night, when her life takes a sudden turn: shes being deported back to Morocco, after being stopped for over speeding, because her Visa expired some time ago.  So in no more than 24 hours, shes deported back to her family and original country. The strong cultural shock and the judgmental differences are pushing the woman to obtain back her place in the city of dreams and her dreams, no matter the costs. But that doesn't mean she will have to return alone, as she finds other things also among her way back to the city. 

After sentence tokenization:
 ['Maya is living the ultimate fashionistas dream: she is working as a stylist for one of the French trend setters, in the capital of haute-couture: Paris.', 'One o

# **Corpus, Vocabulary, Document, Word**

In [6]:
# Number of words in the entire corpus, vocabulary, number of documents
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

raw_corpus = "".join(df['overview'].astype(str))
print("Corpus: \n", raw_corpus,"\n")

lemmatized_corpus = lemma.astype(str).to_list()
print("Preprocessed corpus:\n", lemmatized_corpus)
cv.fit(lemmatized_corpus)
print("Total number of words in the corpus:\t",cv.transform(lemmatized_corpus).sum(),"\n")

#without sklearn
print("Without sklearn")
vocabulary = set(word for doc in lemmatized_corpus for word in str(doc).split())
print("Vocbulary:", vocabulary)
print("Total number of unique words:\t", len(vocabulary),"\n")                              # ,"\n"

#with sklearn
print("With sklearn")
vocab = cv.get_feature_names_out()
print("Vocbulary:", vocab)
print("Total number of unique words:\t", len(vocab),"\n")

print("Number of documents in the corpus:\t", cv.transform(lemmatized_corpus).shape[0])

Corpus: 
 ['maya', 'living', 'ultimate', 'fashionistas', 'dream', 'working', 'stylist', 'one', 'french', 'trend', 'setters', 'capital', 'hautecouture', 'paris', 'one', 'girls', 'fashion', 'shes', 'following', 'dreams', 'one', 'night', 'life', 'takes', 'sudden', 'turn', 'shes', 'deported', 'back', 'morocco', 'stopped', 'speeding', 'visa', 'expired', 'time', 'ago', 'hours', 'shes', 'deported', 'back', 'family', 'original', 'country', 'strong', 'cultural', 'shock', 'judgmental', 'differences', 'pushing', 'woman', 'obtain', 'back', 'place', 'city', 'dreams', 'dreams', 'matter', 'costs', 'doesnt', 'mean', 'return', 'alone', 'finds', 'things', 'also', 'among', 'way', 'back', 'city']['darren', 'taken', 'circus', 'thats', 'chockfull', 'sideshow', 'oddities', 'meets', 'vampire', 'receives', 'lifechanging', 'bite', 'neck']['two', 'veteran', 'new', 'york', 'city', 'detectives', 'work', 'identify', 'possible', 'connection', 'recent', 'murder', 'case', 'believe', 'solved', 'years', 'ago', 'serial',

# OneHotEncoding

In [11]:
from sklearn.preprocessing import OneHotEncoder

corpus_2d = np.array(lemmatized_corpus).reshape(-1,1)
oneHotencode = OneHotEncoder(sparse_output = False)
encoded_array = oneHotencode.fit_transform(corpus_2d)

print("OneHotEncoded Array:\n", encoded_array)

OneHotEncoded Array:
 [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

# Bag of Words

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
BoW = cv.fit_transform(lemmatized_corpus)
feature_names = cv.get_feature_names_out()
BowDf = pd.DataFrame(BoW.toarray(), columns = feature_names)

print(cv.vocabulary_)
print(BowDf)
print(f"Number of times each word has occurred: \n{BowDf.sum(axis=0)}")

{'maya': 215, 'living': 197, 'ultimate': 344, 'fashionistas': 123, 'dream': 105, 'working': 363, 'stylist': 310, 'one': 235, 'french': 140, 'trend': 336, 'setter': 288, 'capital': 51, 'hautecouture': 160, 'paris': 241, 'girl': 149, 'fashion': 122, 'shes': 290, 'following': 132, 'night': 230, 'life': 193, 'take': 317, 'sudden': 314, 'turn': 340, 'deported': 91, 'back': 25, 'morocco': 222, 'stopped': 305, 'speeding': 301, 'visa': 353, 'expired': 115, 'time': 326, 'ago': 4, 'hour': 169, 'family': 120, 'original': 238, 'country': 75, 'strong': 308, 'cultural': 82, 'shock': 291, 'judgmental': 184, 'difference': 94, 'pushing': 264, 'woman': 361, 'obtain': 233, 'place': 251, 'city': 64, 'matter': 213, 'cost': 74, 'doesnt': 102, 'mean': 216, 'return': 277, 'alone': 8, 'find': 128, 'thing': 325, 'also': 10, 'among': 14, 'way': 358, 'darren': 86, 'taken': 318, 'circus': 63, 'thats': 323, 'chockfull': 62, 'sideshow': 295, 'oddity': 234, 'meet': 218, 'vampire': 347, 'receives': 273, 'lifechanging'

# ngrams

In [36]:
from sklearn.feature_extraction.text import CountVectorizer

#unigram
cv = CountVectorizer(ngram_range=(1,1))
unigram = cv.fit_transform(lemmatized_corpus)
print("Unigram:\n",unigram.toarray(),"\n")

#bigram
cv1 = CountVectorizer(ngram_range=(2,2))          #if you want both unigrams and bigrams together, you can use(1,2)
bigram = cv1.fit_transform(lemmatized_corpus)
bigramdf = pd.DataFrame(bigram.toarray(), columns = cv1.get_feature_names_out())
print("Bigram:\n",bigramdf,"\n")

#trigram
cv2 = CountVectorizer(ngram_range = (3,3), max_features = 5000)     #max features is helpful when we are trying to get range of ngrams  -> in this case the vocab may explode, which freezes laptop's memory
trigram = cv2.fit_transform(lemmatized_corpus)
print("Trigram Vocabulary:\t", cv2.vocabulary_)
print("Length of trigram vocabulary:", len(cv2.vocabulary_))   #vocabulary_ -> built in attribute of CV, doesnt work if u dont fit_transform

Unigram:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]] 

Bigram:
     actuality object  adopted girl  advantage leaving  adversary find  \
0                  0             0                  0               0   
1                  0             0                  0               0   
2                  0             0                  0               0   
3                  0             0                  0               0   
4                  0             0                  0               0   
5                  0             0                  0               0   
6                  0             0                  0               0   
7                  0             0                  0               0   
8                  0             0                  0               1   
9                  0             0                  1               0   
10                 0             0                  0   

# TfIdf

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tf = tfidf.fit_transform(lemmatized_corpus).toarray()
tfidf_Df = pd.DataFrame(tf, columns = tfidf.get_feature_names_out())
tfidf_idf_DF = pd.DataFrame(tfidf.idf_, index = tfidf.get_feature_names_out())

print("Tfidf Output:\n", tfidf_Df,"\n")
print("IDF values:\n", tfidf_idf_DF,"\n")

Tfidf Output:
     actuality   adopted  advantage  adversary       ago     alice     alien  \
0    0.000000  0.000000   0.000000    0.00000  0.088625  0.000000  0.000000   
1    0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
2    0.000000  0.000000   0.000000    0.00000  0.185746  0.000000  0.000000   
3    0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
4    0.000000  0.000000   0.000000    0.00000  0.000000  0.251818  0.000000   
5    0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
6    0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
7    0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
8    0.000000  0.000000   0.000000    0.27569  0.000000  0.000000  0.000000   
9    0.000000  0.000000   0.105509    0.00000  0.000000  0.000000  0.000000   
10   0.000000  0.000000   0.000000    0.00000  0.000000  0.000000  0.000000   
11   0.000000  0.000000   0.000000   

# Word2Vec

(Works well with large text data)

In [82]:
import ast
from gensim.models import Word2Vec

tokenized_corpus = [ast.literal_eval(sentence) for sentence in lemmatized_corpus]

model = Word2Vec(window = 10, min_count=1)      #by default mincount will always be 5, vocab reduces
model.build_vocab(tokenized_corpus)
model.train(tokenized_corpus, total_examples = model.corpus_count, epochs = model.epochs)
print(model)

#similarity
print(model.wv.most_similar("young"),"\n")
print(model.wv.doesnt_match(['year', 'time', 'later', 'hour']),"\n")
print(model.wv.similarity('year', 'time'),"\n")

print(model.wv['girl'],"\n")    #vector
print(model.wv.get_normed_vectors(),"\n") #vector representation of the whole corpus


Word2Vec<vocab=375, vector_size=100, alpha=0.025>
[('desert', 0.2738136053085327), ('object', 0.26502302289009094), ('successful', 0.2557450234889984), ('rap', 0.24828891456127167), ('dream', 0.24808505177497864), ('school', 0.2388339787721634), ('however', 0.2362891435623169), ('townsfolk', 0.2276904135942459), ('detective', 0.21334317326545715), ('get', 0.21174347400665283)] 

year 

0.084500276 

[-8.4858984e-03  6.5140235e-03 -5.8102426e-03 -1.7234447e-03
  9.8846818e-04 -1.9638517e-03 -9.0433937e-03  4.0744422e-03
 -6.5923291e-03 -8.7590422e-03  3.9096489e-03 -8.1335043e-04
  6.5019359e-03 -7.0659835e-03  7.9321756e-04 -7.9511607e-04
  5.7348525e-03 -9.7799674e-03  5.9222118e-03 -9.0822661e-03
 -7.7510844e-03  7.9517247e-04 -2.5801258e-03 -3.8791376e-03
 -5.3265062e-03  4.1796970e-03 -9.3961647e-03  1.0845818e-03
  4.0808902e-03 -2.5443014e-04 -4.6219502e-04  7.1331137e-03
 -9.5250681e-03 -9.0362839e-03  4.3892907e-03  8.7773260e-03
 -7.8762863e-03 -8.4074596e-03 -7.1913929e-04 -9

PCA

In [90]:
from sklearn.decomposition import PCA
import plotly.express as px

pca = PCA(n_components = 3)

X = pca.fit_transform(model.wv.get_normed_vectors())
y = model.wv.index_to_key

fig = px.scatter_3d(X[:100], x=0, y=1, z=2, color=y[:100])
fig.show()